In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:,.6f}".format)

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import ID_COL, PROCESSED_DATA_DIR
from src.features_previous import (
    load_previous_application_data,
    preprocess_previous_application,
    aggregate_previous_to_curr,
    build_previous_application_features,
)

In [2]:
base, previous = load_previous_application_data()

print("Base shape:", base.shape)
print("Previous application shape:", previous.shape)
print("Unique SK_ID_CURR in previous:", previous["SK_ID_CURR"].nunique())
print("Unique SK_ID_PREV in previous:", previous["SK_ID_PREV"].nunique())

display(previous.head())

Base shape: (356255, 1)
Previous application shape: (1670214, 37)
Unique SK_ID_CURR in previous: 338857
Unique SK_ID_PREV in previous: 1670214


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,FLAG_LAST_APPL_PER_CONTRACT,NFLAG_LAST_APPL_IN_DAY,RATE_DOWN_PAYMENT,RATE_INTEREST_PRIMARY,RATE_INTEREST_PRIVILEGED,NAME_CASH_LOAN_PURPOSE,NAME_CONTRACT_STATUS,DAYS_DECISION,NAME_PAYMENT_TYPE,CODE_REJECT_REASON,NAME_TYPE_SUITE,NAME_CLIENT_TYPE,NAME_GOODS_CATEGORY,NAME_PORTFOLIO,NAME_PRODUCT_TYPE,CHANNEL_TYPE,SELLERPLACE_AREA,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,"1,730.430000","17,145.000000","17,145.000000",0.000000,"17,145.000000",SATURDAY,15,Y,1,0.000000,0.182832,0.867336,XAP,Approved,-73,Cash through the bank,XAP,NaN,Repeater,Mobile,POS,XNA,Country-wide,35,Connectivity,12.000000,middle,POS mobile with interest,"365,243.000000",-42.000000,300.000000,-42.000000,-37.000000,0.000000
1,2802425,108129,Cash loans,"25,188.615000","607,500.000000","679,671.000000",NaN,"607,500.000000",THURSDAY,11,Y,1,NaN,NaN,NaN,XNA,Approved,-164,XNA,XAP,Unaccompanied,Repeater,XNA,Cash,x-sell,Contact center,-1,XNA,36.000000,low_action,Cash X-Sell: low,"365,243.000000",-134.000000,916.000000,"365,243.000000","365,243.000000",1.000000
2,2523466,122040,Cash loans,"15,060.735000","112,500.000000","136,444.500000",NaN,"112,500.000000",TUESDAY,11,Y,1,NaN,NaN,NaN,XNA,Approved,-301,Cash through the bank,XAP,"Spouse, partner",Repeater,XNA,Cash,x-sell,Credit and cash offices,-1,XNA,12.000000,high,Cash X-Sell: high,"365,243.000000",-271.000000,59.000000,"365,243.000000","365,243.000000",1.000000
3,2819243,176158,Cash loans,"47,041.335000","450,000.000000","470,790.000000",NaN,"450,000.000000",MONDAY,7,Y,1,NaN,NaN,NaN,XNA,Approved,-512,Cash through the bank,XAP,NaN,Repeater,XNA,Cash,x-sell,Credit and cash offices,-1,XNA,12.000000,middle,Cash X-Sell: middle,"365,243.000000",-482.000000,-152.000000,-182.000000,-177.000000,1.000000
4,1784265,202054,Cash loans,"31,924.395000","337,500.000000","404,055.000000",NaN,"337,500.000000",THURSDAY,9,Y,1,NaN,NaN,NaN,Repairs,Refused,-781,Cash through the bank,HC,NaN,Repeater,XNA,Cash,walk-in,Credit and cash offices,-1,XNA,24.000000,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
print(previous.dtypes.value_counts())
print()
print("Top missing columns:")
display(previous.isnull().mean().sort_values(ascending=False).head(20))

object     16
float64    15
int64       6
Name: count, dtype: int64

Top missing columns:


RATE_INTEREST_PRIVILEGED    0.996437
RATE_INTEREST_PRIMARY       0.996437
AMT_DOWN_PAYMENT            0.536365
RATE_DOWN_PAYMENT           0.536365
NAME_TYPE_SUITE             0.491198
NFLAG_INSURED_ON_APPROVAL   0.402981
DAYS_TERMINATION            0.402981
DAYS_LAST_DUE               0.402981
DAYS_LAST_DUE_1ST_VERSION   0.402981
DAYS_FIRST_DUE              0.402981
DAYS_FIRST_DRAWING          0.402981
AMT_GOODS_PRICE             0.230818
AMT_ANNUITY                 0.222867
CNT_PAYMENT                 0.222864
PRODUCT_COMBINATION         0.000207
AMT_CREDIT                  0.000001
NAME_YIELD_GROUP            0.000000
NAME_PORTFOLIO              0.000000
NAME_SELLER_INDUSTRY        0.000000
SELLERPLACE_AREA            0.000000
dtype: float64

In [4]:
previous_clean = preprocess_previous_application(previous)

engineered_cols = [
    "PREV_IS_APPROVED",
    "PREV_IS_REFUSED",
    "PREV_IS_CANCELED",
    "PREV_IS_UNUSED_OFFER",
    "PREV_APP_CREDIT_DIFF",
    "PREV_APP_CREDIT_RATIO",
    "PREV_CREDIT_GOODS_RATIO",
    "PREV_ANNUITY_CREDIT_RATIO",
    "PREV_DOWNPAYMENT_CREDIT_RATIO",
    "PREV_DAYS_DECISION_ABS",
]

display(previous_clean[["SK_ID_CURR", "SK_ID_PREV"] + engineered_cols].head())

,SK_ID_CURR,SK_ID_PREV,PREV_IS_APPROVED,PREV_IS_REFUSED,PREV_IS_CANCELED,PREV_IS_UNUSED_OFFER,PREV_APP_CREDIT_DIFF,PREV_APP_CREDIT_RATIO,PREV_CREDIT_GOODS_RATIO,PREV_ANNUITY_CREDIT_RATIO,PREV_DOWNPAYMENT_CREDIT_RATIO,PREV_DAYS_DECISION_ABS
0,271877,2030495,1,0,0,0,0.000000,1.000000,1.000000,0.100929,0.000000,73
1,108129,2802425,1,0,0,0,"-72,171.000000",0.893815,1.118800,0.037060,NaN,164
2,122040,2523466,1,0,0,0,"-23,944.500000",0.824511,1.212840,0.110380,NaN,301
3,176158,2819243,1,0,0,0,"-20,790.000000",0.955840,1.046200,0.099920,NaN,512
4,202054,1784265,0,1,0,0,"-66,555.000000",0.835282,1.197200,0.079010,NaN,781


In [5]:
decision_summary = pd.DataFrame({
    "approved_rate": [previous_clean["PREV_IS_APPROVED"].mean()],
    "refused_rate": [previous_clean["PREV_IS_REFUSED"].mean()],
    "canceled_rate": [previous_clean["PREV_IS_CANCELED"].mean()],
    "unused_offer_rate": [previous_clean["PREV_IS_UNUSED_OFFER"].mean()],
})

display(decision_summary)

,approved_rate,refused_rate,canceled_rate,unused_offer_rate
0,0.620747,0.174036,0.189388,0.015828


In [6]:
previous_features_only = aggregate_previous_to_curr(previous_clean)

print("Previous features shape:", previous_features_only.shape)
print("Unique SK_ID_CURR:", previous_features_only[ID_COL].nunique())
print("Is unique key:", previous_features_only[ID_COL].is_unique)

display(previous_features_only.head())

Previous features shape: (338857, 39)
Unique SK_ID_CURR: 338857
Is unique key: True


,SK_ID_CURR,PREV_SK_ID_PREV_COUNT,PREV_PREV_IS_APPROVED_SUM,PREV_PREV_IS_REFUSED_SUM,PREV_PREV_IS_CANCELED_SUM,PREV_PREV_IS_UNUSED_OFFER_SUM,PREV_AMT_APPLICATION_MEAN,PREV_AMT_APPLICATION_MAX,PREV_AMT_CREDIT_MEAN,PREV_AMT_CREDIT_MAX,PREV_AMT_ANNUITY_MEAN,PREV_AMT_ANNUITY_MAX,PREV_AMT_DOWN_PAYMENT_MEAN,PREV_AMT_DOWN_PAYMENT_MAX,PREV_AMT_GOODS_PRICE_MEAN,PREV_AMT_GOODS_PRICE_MAX,PREV_CNT_PAYMENT_MEAN,PREV_CNT_PAYMENT_MAX,PREV_RATE_DOWN_PAYMENT_MEAN,PREV_RATE_DOWN_PAYMENT_MAX,PREV_DAYS_DECISION_MEAN,PREV_DAYS_DECISION_MAX,PREV_PREV_DAYS_DECISION_ABS_MEAN,PREV_PREV_DAYS_DECISION_ABS_MIN,PREV_PREV_APP_CREDIT_DIFF_MEAN,PREV_PREV_APP_CREDIT_DIFF_MAX,PREV_PREV_APP_CREDIT_RATIO_MEAN,PREV_PREV_APP_CREDIT_RATIO_MAX,PREV_PREV_CREDIT_GOODS_RATIO_MEAN,PREV_PREV_ANNUITY_CREDIT_RATIO_MEAN,PREV_PREV_DOWNPAYMENT_CREDIT_RATIO_MEAN,PREV_PREV_APPROVED_AMT_CREDIT_MEAN,PREV_PREV_APPROVED_AMT_CREDIT_MAX,PREV_PREV_APPROVED_AMT_ANNUITY_MEAN,PREV_PREV_APPROVED_AMT_ANNUITY_MAX,PREV_PREV_APPROVED_APP_CREDIT_RATIO_MEAN,PREV_APPROVAL_RATE,PREV_REFUSAL_RATE,PREV_CANCEL_RATE
0,100001,1,1,0,0,0,"24,835.500000","24,835.500000","23,787.000000","23,787.000000","3,951.000000","3,951.000000","2,520.000000","2,520.000000","24,835.500000","24,835.500000",8.000000,8.000000,0.104326,0.104326,"-1,740.000000",-1740,"1,740.000000",1740,"1,048.500000","1,048.500000",1.044079,1.044079,0.957782,0.166099,0.105940,"23,787.000000","23,787.000000","3,951.000000","3,951.000000",1.044079,1.000000,0.000000,0.000000
1,100002,1,1,0,0,0,"179,055.000000","179,055.000000","179,055.000000","179,055.000000","9,251.775000","9,251.775000",0.000000,0.000000,"179,055.000000","179,055.000000",24.000000,24.000000,0.000000,0.000000,-606.000000,-606,606.000000,606,0.000000,0.000000,1.000000,1.000000,1.000000,0.051670,0.000000,"179,055.000000","179,055.000000","9,251.775000","9,251.775000",1.000000,1.000000,0.000000,0.000000
2,100003,3,3,0,0,0,"435,436.500000","900,000.000000","484,191.000000","1,035,882.000000","56,553.990000","98,356.995000","3,442.500000","6,885.000000","435,436.500000","900,000.000000",10.000000,12.000000,0.050030,0.100061,"-1,305.000000",-746,"1,305.000000",746,"-48,754.500000",756.000000,0.949329,1.011109,1.057664,0.126383,0.050585,"484,191.000000","1,035,882.000000","56,553.990000","98,356.995000",0.949329,1.000000,0.000000,0.000000
3,100004,1,1,0,0,0,"24,282.000000","24,282.000000","20,106.000000","20,106.000000","5,357.250000","5,357.250000","4,860.000000","4,860.000000","24,282.000000","24,282.000000",4.000000,4.000000,0.212008,0.212008,-815.000000,-815,815.000000,815,"4,176.000000","4,176.000000",1.207699,1.207699,0.828021,0.266450,0.241719,"20,106.000000","20,106.000000","5,357.250000","5,357.250000",1.207699,1.000000,0.000000,0.000000
4,100005,2,1,0,1,0,"22,308.750000","44,617.500000","20,076.750000","40,153.500000","4,813.200000","4,813.200000","4,464.000000","4,464.000000","44,617.500000","44,617.500000",12.000000,12.000000,0.108964,0.108964,-536.000000,-315,536.000000,315,"2,232.000000","4,464.000000",1.111173,1.111173,0.899950,0.119870,0.111173,"40,153.500000","40,153.500000","4,813.200000","4,813.200000",1.111173,0.500000,0.000000,0.500000


In [7]:
previous_features = build_previous_application_features(save=True)

print("Final previous_application features shape:", previous_features.shape)
print("Unique SK_ID_CURR:", previous_features[ID_COL].nunique())
print("Is unique key:", previous_features[ID_COL].is_unique)

display(previous_features.head())

Final previous_application features shape: (356255, 39)
Unique SK_ID_CURR: 356255
Is unique key: True


,SK_ID_CURR,PREV_SK_ID_PREV_COUNT,PREV_PREV_IS_APPROVED_SUM,PREV_PREV_IS_REFUSED_SUM,PREV_PREV_IS_CANCELED_SUM,PREV_PREV_IS_UNUSED_OFFER_SUM,PREV_AMT_APPLICATION_MEAN,PREV_AMT_APPLICATION_MAX,PREV_AMT_CREDIT_MEAN,PREV_AMT_CREDIT_MAX,PREV_AMT_ANNUITY_MEAN,PREV_AMT_ANNUITY_MAX,PREV_AMT_DOWN_PAYMENT_MEAN,PREV_AMT_DOWN_PAYMENT_MAX,PREV_AMT_GOODS_PRICE_MEAN,PREV_AMT_GOODS_PRICE_MAX,PREV_CNT_PAYMENT_MEAN,PREV_CNT_PAYMENT_MAX,PREV_RATE_DOWN_PAYMENT_MEAN,PREV_RATE_DOWN_PAYMENT_MAX,PREV_DAYS_DECISION_MEAN,PREV_DAYS_DECISION_MAX,PREV_PREV_DAYS_DECISION_ABS_MEAN,PREV_PREV_DAYS_DECISION_ABS_MIN,PREV_PREV_APP_CREDIT_DIFF_MEAN,PREV_PREV_APP_CREDIT_DIFF_MAX,PREV_PREV_APP_CREDIT_RATIO_MEAN,PREV_PREV_APP_CREDIT_RATIO_MAX,PREV_PREV_CREDIT_GOODS_RATIO_MEAN,PREV_PREV_ANNUITY_CREDIT_RATIO_MEAN,PREV_PREV_DOWNPAYMENT_CREDIT_RATIO_MEAN,PREV_PREV_APPROVED_AMT_CREDIT_MEAN,PREV_PREV_APPROVED_AMT_CREDIT_MAX,PREV_PREV_APPROVED_AMT_ANNUITY_MEAN,PREV_PREV_APPROVED_AMT_ANNUITY_MAX,PREV_PREV_APPROVED_APP_CREDIT_RATIO_MEAN,PREV_APPROVAL_RATE,PREV_REFUSAL_RATE,PREV_CANCEL_RATE
0,100002,1.000000,1.000000,0.000000,0.000000,0.000000,"179,055.000000","179,055.000000","179,055.000000","179,055.000000","9,251.775000","9,251.775000",0.000000,0.000000,"179,055.000000","179,055.000000",24.000000,24.000000,0.000000,0.000000,-606.000000,-606.000000,606.000000,606.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.051670,0.000000,"179,055.000000","179,055.000000","9,251.775000","9,251.775000",1.000000,1.000000,0.000000,0.000000
1,100003,3.000000,3.000000,0.000000,0.000000,0.000000,"435,436.500000","900,000.000000","484,191.000000","1,035,882.000000","56,553.990000","98,356.995000","3,442.500000","6,885.000000","435,436.500000","900,000.000000",10.000000,12.000000,0.050030,0.100061,"-1,305.000000",-746.000000,"1,305.000000",746.000000,"-48,754.500000",756.000000,0.949329,1.011109,1.057664,0.126383,0.050585,"484,191.000000","1,035,882.000000","56,553.990000","98,356.995000",0.949329,1.000000,0.000000,0.000000
2,100004,1.000000,1.000000,0.000000,0.000000,0.000000,"24,282.000000","24,282.000000","20,106.000000","20,106.000000","5,357.250000","5,357.250000","4,860.000000","4,860.000000","24,282.000000","24,282.000000",4.000000,4.000000,0.212008,0.212008,-815.000000,-815.000000,815.000000,815.000000,"4,176.000000","4,176.000000",1.207699,1.207699,0.828021,0.266450,0.241719,"20,106.000000","20,106.000000","5,357.250000","5,357.250000",1.207699,1.000000,0.000000,0.000000
3,100006,9.000000,5.000000,1.000000,3.000000,0.000000,"272,203.260000","688,500.000000","291,695.500000","906,615.000000","23,651.175000","39,954.510000","34,840.170000","66,987.000000","408,304.890000","688,500.000000",23.000000,48.000000,0.163412,0.217830,-272.444444,-181.000000,272.444444,181.000000,"-19,492.240000","66,987.000000",1.010763,1.250017,1.012684,0.069304,0.180612,"343,728.900000","675,000.000000","21,842.190000","39,954.510000",1.061032,0.555556,0.111111,0.333333
4,100007,6.000000,6.000000,0.000000,0.000000,0.000000,"150,530.250000","247,500.000000","166,638.750000","284,400.000000","12,278.805000","22,678.785000","3,390.750000","3,676.500000","150,530.250000","247,500.000000",20.666667,48.000000,0.159516,0.218890,"-1,222.833333",-374.000000,"1,222.833333",374.000000,"-16,108.500000","2,560.500000",0.969650,1.175185,1.046356,0.090659,0.176401,"166,638.750000","284,400.000000","12,278.805000","22,678.785000",0.969650,1.000000,0.000000,0.000000


In [8]:
prev_cols = [c for c in previous_features.columns if c != ID_COL]

display(
    previous_features[prev_cols]
    .isnull()
    .mean()
    .sort_values(ascending=False)
    .head(20)
)

PREV_AMT_DOWN_PAYMENT_MAX                  0.105267
PREV_PREV_DOWNPAYMENT_CREDIT_RATIO_MEAN    0.105267
PREV_RATE_DOWN_PAYMENT_MEAN                0.105267
PREV_AMT_DOWN_PAYMENT_MEAN                 0.105267
PREV_RATE_DOWN_PAYMENT_MAX                 0.105267
PREV_PREV_APPROVED_APP_CREDIT_RATIO_MEAN   0.052114
PREV_PREV_APPROVED_AMT_ANNUITY_MAX         0.052095
PREV_PREV_APPROVED_AMT_ANNUITY_MEAN        0.052095
PREV_PREV_APPROVED_AMT_CREDIT_MAX          0.052089
PREV_PREV_APPROVED_AMT_CREDIT_MEAN         0.052089
PREV_PREV_CREDIT_GOODS_RATIO_MEAN          0.051938
PREV_AMT_GOODS_PRICE_MEAN                  0.051822
PREV_AMT_GOODS_PRICE_MAX                   0.051822
PREV_PREV_ANNUITY_CREDIT_RATIO_MEAN        0.050186
PREV_AMT_ANNUITY_MAX                       0.050183
PREV_AMT_ANNUITY_MEAN                      0.050183
PREV_CNT_PAYMENT_MEAN                      0.050178
PREV_CNT_PAYMENT_MAX                       0.050178
PREV_PREV_APP_CREDIT_RATIO_MAX             0.049543
PREV_PREV_AP

In [9]:
check_cols = [
    "PREV_SK_ID_PREV_COUNT",
    "PREV_PREV_IS_APPROVED_SUM",
    "PREV_PREV_IS_REFUSED_SUM",
    "PREV_APPROVAL_RATE",
    "PREV_REFUSAL_RATE",
    "PREV_CANCEL_RATE",
]

existing_check_cols = [c for c in check_cols if c in previous_features.columns]
display(previous_features[existing_check_cols].describe().T)

,count,mean,std,min,25%,50%,75%,max
PREV_SK_ID_PREV_COUNT,"338,857.000000",4.928964,4.220716,1.000000,2.000000,4.000000,7.000000,77.000000
PREV_PREV_IS_APPROVED_SUM,"338,857.000000",3.059642,2.135404,0.000000,1.000000,3.000000,4.000000,27.000000
PREV_PREV_IS_REFUSED_SUM,"338,857.000000",0.857819,1.830574,0.000000,0.000000,0.000000,1.000000,68.000000
PREV_APPROVAL_RATE,"338,857.000000",0.744495,0.263250,0.000000,0.500000,0.777778,1.000000,1.000000
PREV_REFUSAL_RATE,"338,857.000000",0.111420,0.183730,0.000000,0.000000,0.000000,0.200000,1.000000
PREV_CANCEL_RATE,"338,857.000000",0.129163,0.188675,0.000000,0.000000,0.000000,0.250000,1.000000
